Import all modules and classes


In [4]:
import os
from pprint import pprint

import requests
from dotenv import load_dotenv
from langchain.agents import create_agent
import langchain
#print(langchain.__version__)

#load env
load_dotenv()

True

Define a tool for checking bible verse based on topics 

In [5]:
def fetch_bible_verse(reference: str)->str:
    """ Get the current bible text based on study verse."""

    # OpenBibleVerse API endpoint
    url = f"https://bible-api.com/{reference}"

     # Send the GET http request
    response = requests.get(url, timeout=10)

    # Check if the request was successful
    if response.status_code != 200:
        return f"Could not get bible text for {reference}. try another reference: {response.text}"
   
    # Parse the JSON response
    data = response.json()

    text = data["text"]



     # Return the bible text information as a string
    return (
        f" {reference} reads :{text} : "
        
    )


Create the agent

In [6]:

# Create the agent using the OpenAI API
"""
Note: The OpenAI API key is stored in the .env file and it's auto-detected by LangChain/OpenAI.
How it works:
- Python reads the .env file with the aid of the `load_dotenv()` function.
- Variables inside it become available as environment variables.
- LangChain/OpenAI automatically checks for OPENAI_API_KEY. So we don't need to pass it explicitly.
"""


graph = create_agent(
    model="groq:llama-3.3-70b-versatile",
    tools=[fetch_bible_verse],
    system_prompt="You are a Bible study assistant helping teachers prepare lessons. " \
    "When the user gives you a topic, propose 5–7 candidate verse references that address it, then use the fetch_bible_verse tool to retrieve each one." \
    " After reading the actual text returned by the tool, keep only verses that genuinely fit the topic (don't force-fit tangential ones)." \
    " If a reference comes back as not found, try a different one. Return 3–5 final verses with the reference, the verse text, and one sentence explaining why it fits the topic."
)

In [49]:
inputs = {
    "messages": [
        {
            "role": "user",
            "content" : "What does Isaiah 66:25 say?"
        }
    ]
}

In [7]:
inputs = {
    "messages": [
        {
            "role": "user",
            "content" : "What does John 3:16 say?"
        }
    ]
}

In [40]:
inputs = {
    "messages": [
        {
            "role": "user",
            "content": "I want to teach a lesson about forgiveness. Find me 3-5 strong verses for this topic."
        }
    ]
}

In [43]:
inputs = {
    "messages" : [
        {
            "role": "user",
            "content": "These are recommended verses on forgiveness, what does these verses say : Matthew 6:14-15, Luke 6:37-38,Ephesians 4:32, Colossians 3:13, Mark 11:25"
        }
    ]
}

Run the agent and parse the agent's output while streaming.

In [8]:
# Streaming helps us get the output in real-time as it's generated
for chunk in graph.stream(inputs, stream_mode="updates"):
    # 1. Model output
    if "model" in chunk:
        messages = chunk["model"]["messages"]

        for msg in messages:
            # Tool call request
            if msg.tool_calls:
                print("\n🤖 Agent wants to use a tool:")
                for tool_call in msg.tool_calls:
                    print(f"Tool: {tool_call['name']}")
                    print(f"Arguments: {tool_call['args']}")

            # Final AI response
            if msg.content:
                print("\n✅ Final Answer:")
                print(msg.content)

            # Token usage
            if hasattr(msg, "usage_metadata") and msg.usage_metadata:
                print("\n📊 Token Usage:")
                pprint(msg.usage_metadata)

    # 2. Tool output
    if "tools" in chunk:
        messages = chunk["tools"]["messages"]

        for msg in messages:
            print("\n🛠️ Tool Result:")
            print(f"Tool: {msg.name}")
            print(f"Output: {msg.content}")


🤖 Agent wants to use a tool:
Tool: fetch_bible_verse
Arguments: {'reference': 'John 3:16'}

📊 Token Usage:
{'input_tokens': 344, 'output_tokens': 22, 'total_tokens': 366}

🛠️ Tool Result:
Tool: fetch_bible_verse
Output:  John 3:16 reads :
For God so loved the world, that he gave his one and only Son, that whoever believes in him should not perish, but have eternal life.

 : 

✅ Final Answer:
John 3:16 says, "For God so loved the world that he gave his one and only Son, that whoever believes in him shall not perish but have eternal life." This verse fits the topic of God's love because it clearly states the depth of God's love for the world and the sacrifice He made to save humanity.

📊 Token Usage:
{'input_tokens': 415, 'output_tokens': 68, 'total_tokens': 483}
